# Train / Test Split

Partimos de un dataset de LendingClub con **~1.76 millones** de préstamos y 151 variables. Vamos a generar **3 datasets**:

| Dataset | Filas | Distribución de `loan_status` | Uso |
|---|---|---|---|
| `df_train_small` | 80.000 | ~80/20 (original, desbalanceada) | Entrenar modelo A |
| `df_train_balanced` | 80.000 | 50/50 (balanceada) | Entrenar modelo B |
| `df_test_small` | 20.000 | ~80/20 (original, desbalanceada) | Evaluar ambos modelos |

Tener dos sets de entrenamiento (uno desbalanceado y otro balanceado) nos permitirá comparar cómo afecta el balanceo de clases al rendimiento predictivo.

In [1]:
import pandas as pd

## Cargo el dataset completo

In [ ]:
df = pd.read_csv("data/accepted_2007_to_2017.csv")
df.shape

/var/folders/90/nx4mc_9d0wv10t5rvcldvb940000gn/T/ipykernel_47739/4131002735.py:2: DtypeWarning: Columns (0: desc, 1: next_pymnt_d, 2: verification_status_joint, 3: sec_app_earliest_cr_line, 4: hardship_type, 5: hardship_reason, 6: hardship_status, 7: hardship_start_date, 8: hardship_end_date, 9: payment_plan_start_date, 10: hardship_loan_status) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/accepted_2007_to_2017.csv")


(1765426, 151)

In [3]:
df["loan_status"].value_counts(normalize=True)*100

loan_status
Fully Paid                                             58.303605
Current                                                25.553946
Charged Off                                            14.709877
Late (31-120 days)                                      0.806944
In Grace Period                                         0.312502
Late (16-30 days)                                       0.155543
Does not meet the credit policy. Status:Fully Paid      0.112607
Does not meet the credit policy. Status:Charged Off     0.043106
Default                                                 0.001869
Name: proportion, dtype: float64

## Exploración de la variable objetivo

In [16]:
df["loan_status"].value_counts(normalize=True)*100

loan_status
Fully Paid                                             58.303605
Current                                                25.553946
Charged Off                                            14.709877
Late (31-120 days)                                      0.806944
In Grace Period                                         0.312502
Late (16-30 days)                                       0.155543
Does not meet the credit policy. Status:Fully Paid      0.112607
Does not meet the credit policy. Status:Charged Off     0.043106
Default                                                 0.001869
Name: proportion, dtype: float64

## Filtrado

In [4]:
# Solo nos interesan los estados terminales: Fully Paid (pagado) y Charged Off (impago).
df_filtered = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])]
df_filtered.shape

(1288999, 151)

In [5]:
df_filtered["loan_status"].value_counts(normalize=True) * 100

loan_status
Fully Paid     79.853204
Charged Off    20.146796
Name: proportion, dtype: float64

## Split train/test (80/20)

In [ ]:
# train/test split
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df_filtered, test_size=0.2, random_state=42)

In [9]:
df_train.shape

(1031199, 151)

In [10]:
df_test.shape

(257800, 151)

## Submuestreo de test: 20.000 filas
Del conjunto de test (~258K filas) extraemos una muestra aleatoria de **20.000 filas** para trabajar con un tamaño manejable.

In [ ]:
# No podemos balancear test -- cojo una muestra aleatoria de 20.000 filas de test y la balanceo
df_test_small = df_test.sample(20_000, random_state=42)
df_test_small.shape

(20000, 151)

In [15]:
df_test_small["loan_status"].value_counts(normalize=True) * 100

loan_status
Fully Paid     80.015
Charged Off    19.985
Name: proportion, dtype: float64

## Submuestreo de train desbalanceado: 80.000 filas
Del conjunto de train (~1.03M filas) extraemos **80.000 muestras** aleatorias. Esta muestra conserva la distribución original ~80/20 ya que es una muestra aleatoria simple.
Este será el **primer set de entrenamiento** (desbalanceado), que refleja la proporción real de pagos e impagos.

In [17]:
df_train_small = df_train.sample(80_000, random_state=42)

In [18]:
df_train_small["loan_status"].value_counts(normalize=True) * 100

loan_status
Fully Paid     79.705
Charged Off    20.295
Name: proportion, dtype: float64

## Creación del train balanceado: 80.000 filas (50/50)

In [19]:
# Una muestra de train balanceada con respecto a loan_status

df_train_full_paid = df_train[df_train["loan_status"] == "Fully Paid"].sample(40_000, random_state=42)
df_train_charged_off = df_train[df_train["loan_status"] == "Charged Off"].sample(40_000, random_state=42)

df_train_balanced = pd.concat([df_train_full_paid, df_train_charged_off]) 

df_train_balanced = df_train_balanced.sample(frac=1, random_state=42)

In [21]:
# Compruebo que está balanceado
df_train_balanced["loan_status"].value_counts(normalize=True) * 100

loan_status
Charged Off    50.0
Fully Paid     50.0
Name: proportion, dtype: float64

## Guardado de los 3 datasets finales
Exportamos los tres DataFrames a CSV:

| Archivo | Contenido | Filas | Distribución |
|---|---|---|---|
| `df_train_small.csv` | Train desbalanceado | 80.000 | ~80/20 |
| `df_train_balanced.csv` | Train balanceado | 80.000 | 50/50 |
| `df_test_small.csv` | Test | 20.000 | ~80/20 |

In [22]:
# Guardar df_train_balanced, df_test_small y df_train_small to csv
df_train_balanced.to_csv("data/df_train_balanced.csv", index=False) 
df_test_small.to_csv("data/df_test_small.csv", index=False) 
df_train_small.to_csv("data/df_train_small.csv", index=False)